### Классификация для CC50 > медианы



### Опиание

Для классификации CC50 был использован предварительно обработанный десериализированный датасет.

Дополнительно был произведен отбор целевой переменной, признаков, разделение набора данных на обучающую и тестовую выборку.

Далее для модели LogisticRegression было произведено масштабирование данных.

Для классификации были выбраны 3 модели: LogisticRegression в качестве baseline, RandomForestClassifier, CatBoostClassifier в качестве основных моделей.

Для каждой модели был реализован подбор параметров с помощью GridSearchCV, а также кросс-валидация.

Также были рассчитаны метрики f1_score, roc_auc_score

| Модель                | f1_score | roc_auc_score|
|-----------------------|----------|--------------|
| LogisticRegression    | 0.73     | 0.82         |
| RandomForestClassifier| 0.77     | 0.86         |
| CatBoostClassifier    | 0.76     | 0.87         |

По результатам все модели показали сопоставимое качество по метрике F1-score (0.74/0.77/0.76), что свидетельствует о хорошем балансе между precision и recall и способности моделей корректно классифицировать объекты.

Модели ансамблей (RandomForest и CatBoost) превосходят линейную модель LogisticRegression по способности различать классы, о чем говорит метрика roc_auc_score, что указывает на наличие нелинейных зависимостей в данных, которые линейная модель улавливает хуже.

In [1]:
# Устанавливаем библиотеку
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 6.5 MB/s eta 0:00:00


In [2]:
# Импортируем необходимые библиотеки
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from catboost import CatBoostClassifier

from sklearn.metrics import f1_score, roc_auc_score

In [3]:
# Загружаем DataFrame из файла
df = pd.read_pickle('my_dataset.pkl')

print(df.head())

     IC50, mM    CC50, mM         SI  MaxAbsEStateIndex  MinAbsEStateIndex  \
0    6.239374  175.482382  28.125000           5.094096           0.387225   
1    0.771831    5.402819   7.000000           3.961417           0.533868   
2  223.808778  161.142320   0.720000           2.627117           0.543231   
3    1.705624  107.855654  63.235294           5.097360           0.390603   
4  107.131532  139.270991   1.300000           5.150510           0.270476   

   MinEStateIndex       qed        SPS    MolWt  HeavyAtomMolWt  ...  \
0        0.387225  0.417362  42.928571  384.652         340.300  ...   
1        0.533868  0.462473  45.214286  388.684         340.300  ...   
2        0.543231  0.260923  42.187500  446.808         388.344  ...   
3        0.390603  0.377846  41.862069  398.679         352.311  ...   
4        0.270476  0.429038  36.514286  466.713         424.377  ...   

   fr_tetrazole  fr_thiazole  fr_thiocyan  fr_thiophene  fr_unbrch_alkane  \
0             0      

In [4]:
# Выделение признаков и целевых переменных для классификации
y = df['CC50_above_median']

X = df.drop(['IC50, mM', 'CC50, mM', 'SI', 'IC50_above_median', 'CC50_above_median', 'SI_above_median', 'SI_above_8'], axis=1)

In [5]:
# Разделяем данные на обучающую и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Обучающая выборка: {X_train.shape[0]} строк")
print(f"Тестовая выборка: {X_test.shape[0]} строк")

Обучающая выборка: 764 строк
Тестовая выборка: 191 строк


In [6]:
# Производим масштабирование данных для логистической регрессии
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [7]:
# Обучаем LogisticRegression с подбором параметров, кросс-валидацией и расчетом метрик
param_grid_LR = {
    "C": [0.01, 0.1, 1, 10, 100],
    "penalty": ["l2"],
    "solver": ["lbfgs"],
    }


grid_LR = GridSearchCV(
    LogisticRegression(max_iter=1000),
    param_grid_LR,
    cv=3,
    scoring="roc_auc"
)

grid_LR.fit(X_train_scaled, y_train)

best_model_LR = grid_LR.best_estimator_
y_pred_LR = best_model_LR.predict(X_test_scaled)
y_proba_LR = best_model_LR.predict_proba(X_test_scaled)[:, 1]

roc_auc_LR = roc_auc_score(y_test, y_proba_LR)
f1_LR = f1_score(y_test, y_pred_LR)

print("ROC-AUC:", roc_auc_LR)
print("F1:", f1_LR)

ROC-AUC: 0.8247807017543861
F1: 0.7346938775510204


In [8]:
# Обучаем RandomForestClassifier с подбором параметров, кросс-валидацией и расчетом метрик
param_grid_RF = {
    "n_estimators": [300, 350],
    "max_depth": [10, 11],
    "min_samples_split": [4, 5],
    "min_samples_leaf": [1, 2],
    "max_features": ["sqrt", "log2"],
    }

grid_RF = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid_RF,
    cv=3,
    scoring="roc_auc",
    n_jobs=-1
)

grid_RF.fit(X_train, y_train)

best_model_RF = grid_RF.best_estimator_
y_pred_RF = best_model_RF.predict(X_test)
y_proba_RF = best_model_RF.predict_proba(X_test)[:, 1]

roc_auc_RF = roc_auc_score(y_test, y_proba_RF)
f1_RF = f1_score(y_test, y_pred_RF)

print("ROC-AUC:", roc_auc_RF)
print("F1:", f1_RF)

ROC-AUC: 0.8608552631578947
F1: 0.7668393782383419


In [9]:
# Обучаем CatBoostClassifier с подбором параметров, кросс-валидацией и расчетом метрик
param_grid_CB = {
    "iterations": [250, 300],
    "depth": [7, 8],
    "learning_rate": [0.03, 0.05],
    "l2_leaf_reg": [4, 5]
}

grid_CB = GridSearchCV(
    CatBoostClassifier(
        random_state=42,
        verbose=0,
        loss_function="Logloss"
    ),
    param_grid_CB,
    cv=3,
    scoring="roc_auc",
    n_jobs=-1
)

grid_CB.fit(X_train, y_train)

best_model_CB = grid_CB.best_estimator_

y_pred_CB = best_model_CB.predict(X_test)
y_proba_CB = best_model_CB.predict_proba(X_test)[:, 1]

roc_auc_CB = roc_auc_score(y_test, y_proba_CB)
f1_CB = f1_score(y_test, y_pred_CB)

print("Best params:", grid_CB.best_params_)
print("Best CV ROC-AUC:", grid_CB.best_score_)
print("ROC-AUC:", roc_auc_CB)
print("F1:", f1_CB)

/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Best params: {'depth': 7, 'iterations': 250, 'l2_leaf_reg': 5, 'learning_rate': 0.03}
Best CV ROC-AUC: 0.80498635997272
ROC-AUC: 0.8663377192982455
F1: 0.7614213197969543
